# Hands-on: Simple RAG from Scratch!

Nama : Tika Altiora Sitanggang

NIM  : 42324025

# 1. Setup & Install Library

Sebelum menjalankan notebook ini, pastikan runtime sudah diset ke **GPU (T4)**: Runtime --> Change runtime type --> T4 GPU --> Save

In [4]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ModuleNotFoundError: No module named 'torch'

Penjelasan: Kode ini mengimpor library `torch` (PyTorch), lalu mengecek apakah GPU tersedia lewat `torch.cuda.is_available()`. Hasilnya disimpan di variabel `DEVICE`. Isinya `"cuda"` kalau GPU ada, dan `"cpu"` kalau tidak ada.

Variabel `DEVICE` dipakai lagi di bagian load model untuk menentukan tipe data model. Pengecekan ini penting karena model LLM jauh lebih cepat dijalankan di GPU (T4) daripada di CPU.

In [ ]:
!pip install -q \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    transformers \
    accelerate

Penjelasan: Perintah `!pip install` dipakai untuk memasang semua library yang dibutuhkan. Tanda `!` artinya perintah dijalankan di terminal Colab, bukan sebagai kode Python. Opsi `-q` membuat proses instalasi tidak menampilkan log panjang, dan tanda `\` hanya untuk melanjutkan perintah ke baris berikutnya supaya lebih rapi.

Fungsi tiap library:
1. `langchain-huggingface`: penghubung antara LangChain dan model dari HuggingFace.
2. `langchain-text-splitters`: berisi tool untuk memecah teks panjang menjadi chunk.
3. `faiss-cpu`: vector database untuk menyimpan dan mencari vektor embedding. Versi CPU dipilih karena cukup untuk data sekecil ini.
4. `sentence-transformers`: untuk memuat embedding model yang mengubah teks menjadi vektor.
5. `pypdf`: untuk membaca dan mengekstrak teks dari file PDF.
6. `transformers`: library utama HuggingFace untuk memuat dan menjalankan model LLM.
7. `accelerate`: membantu proses load model ke GPU. Library ini dibutuhkan agar parameter `device_map="auto"` bisa berjalan.

# 2. Setup HuggingFace Token

HuggingFace Token dibutuhkan untuk mengakses model dan mendownload weights dari **HuggingFace Hub**.

**Cara mendapatkan token gratis:**
1. Buka https://huggingface.co/settings/tokens
2. Klik **"+ Create new token"**
3. Pilih token type: **Read**
4. Buat nama token, lalu klik **Create token**
5. Copy dan simpan tokennya

**Cara setup HuggingFace Token di Google Colab dengan Secrets:**
> Lebih aman karena token tidak terlihat secara langsung di notebook
1. Klik ikon 🔑 di sidebar kiri Colab
2. Klik **"Add new secret"**
3. Name: `HF_TOKEN`, Value: paste token dari HuggingFace
4. Aktifkan toggle **"Notebook access"**

In [ ]:
import os
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

# Set sebagai environment variable agar otomatis dikenali transformers & HF Hub
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN
print(f"HF Token prefix: {HF_TOKEN[:8]}...")

ModuleNotFoundError: No module named 'google.colab'

Penjelasan: Kode ini mengambil token HuggingFace dari Colab Secrets, lalu mendaftarkannya supaya bisa dipakai library lain.

1. `userdata.get('HF_TOKEN')` membaca secret bernama `HF_TOKEN` yang sudah disimpan di sidebar Colab. Dengan cara ini token tidak perlu ditulis langsung di dalam kode.
2. `os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN` menyimpan token sebagai environment variable. Library `transformers` dan HuggingFace Hub otomatis mencari token dengan nama ini, jadi kita tidak perlu memasukkannya lagi saat download model.
3. `print(f"HF Token prefix: {HF_TOKEN[:8]}...")` hanya menampilkan 8 karakter pertama token. Tujuannya sebagai tanda bahwa token berhasil terbaca, tanpa memperlihatkan token secara utuh.

Output `hf_cMEKG...` menandakan token berhasil dimuat.

# 3. Upload & Load Dokumen PDF

Tahap pertama pipeline RAG: membaca isi dokumen PDF dan mengubahnya menjadi
objek `Document` yang bisa diproses LangChain.

> Format lain (`.txt`, `.docx`, `.csv`) juga didukung LangChain, namun membutuhkan loader yang berbeda.

In [ ]:
from google.colab import files
from pypdf import PdfReader
from langchain_core.documents import Document

# Upload PDF
print("Pilih file PDF ...")
uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]
print(f"File PDF berhasil diupload: {pdf_filename}")

# Load & ekstrak PDF
# Setiap halaman dijadikan satu objek Document
# dengan metadata page number & nama file sebagai sumbernya
reader = PdfReader(pdf_filename)
documents = []
for page_num, page in enumerate(reader.pages):
    text = page.extract_text()
    if text and text.strip():
        documents.append(Document(
            page_content=text,
            metadata={"page": page_num, "source": pdf_filename}
        ))

# Preview PDF
print(f"\nFile PDF berhasil diload!")
print(f"   Total halaman       : {len(reader.pages)}")
print(f"   Halaman dengan teks : {len(documents)}")
print(f"\nContoh isi halaman pertama (200 karakter pertama):")
print(f" '{documents[0].page_content[:200]}...'")

Pilih file PDF ...


Saving Accountability_42324025.pdf to Accountability_42324025 (1).pdf
File PDF berhasil diupload: Accountability_42324025 (1).pdf

File PDF berhasil diload!
   Total halaman       : 9
   Halaman dengan teks : 9

Contoh isi halaman pertama (200 karakter pertama):
 'Laporan Praktikum Socio - Informatika dan Profesionalisme 
“Accountability and Workplace Relationships” 
 
 
 
Nama : Tika Altiora Sitanggang 
NIM : 42324025 
Prodi : D-III Teknologi Informasi 
 
 
 
...'


Penjelasan: Kode ini bertugas membaca file PDF dan mengubahnya menjadi daftar objek `Document` milik LangChain.

1. `files.upload()` membuka kotak upload di Colab. Hasilnya berupa dictionary yang key-nya nama file, lalu `list(uploaded.keys())[0]` mengambil nama file pertama.
2. `PdfReader(pdf_filename)` membuka PDF supaya halamannya bisa dibaca satu per satu.
3. Perulangan `for page_num, page in enumerate(reader.pages)` mengambil teks tiap halaman dengan `page.extract_text()`.
4. Kondisi `if text and text.strip()` memastikan halaman kosong atau halaman yang isinya cuma gambar dilewati.
5. Tiap halaman dibungkus jadi satu `Document` dengan dua bagian. `page_content` berisi teks halamannya, dan `metadata` berisi nomor halaman serta nama file sumber. Metadata ini nanti berguna untuk menampilkan asal informasi dari jawaban chatbot.

Dari output, PDF yang diupload punya 9 halaman dan semuanya punya teks, jadi ada 9 objek `Document`. Nama file berubah menjadi `(1)` karena di Colab sudah ada file dengan nama yang sama.

# 4. Chucking

Dokumen yang panjang tidak bisa langsung diproses LLM karena adanya batasan
**context window** (jumlah token maksimum yang bisa diproses sekaligus).
Chunking memecah dokumen menjadi potongan-potongan kecil yang lebih mudah dicari
dan dimasukkan ke dalam prompt.

**Parameter chunking:**
- `chunk_size` — jumlah karakter per chunk
- `chunk_overlap` — jumlah karakter yang overlap antar chunk

**Kenapa perlu parameter overlap?**
Tanpa parameter overlap, informasi yang berada di batas antar chunk bisa terpotong dan hilang. Parameter Overlap memastikan koneksi antar kalimat tetap terjaga.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Parameter chunking
# chunk_size    : panjang maksimum tiap chunk (dalam karakter)
# chunk_overlap : jumlah karakter yang overlap antar chunk
# separators    : urutan pemisah yang dicoba.dari yang paling diutamakan
#                 ke yang paling akhir (karakter tunggal / kosong)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Chuck dokumen
chunks = text_splitter.split_documents(documents)

# Preview hasil Chucking
print(f"Hasil chunking:")
print(f"   Total halaman asli : {len(documents)}")
print(f"   Total chunks       : {len(chunks)}")
print(f"   Rata-rata panjang  : {sum(len(c.page_content) for c in chunks) // len(chunks)} karakter")
print(f"\nContoh chunk pertama:")
print(f"   Metadata : {chunks[0].metadata}")
print(f"   Konten   : '{chunks[0].page_content[:200]}...'")

Hasil chunking:
   Total halaman asli : 9
   Total chunks       : 36
   Rata-rata panjang  : 422 karakter

Contoh chunk pertama:
   Metadata : {'page': 0, 'source': 'Accountability_42324025 (1).pdf'}
   Konten   : 'Laporan Praktikum Socio - Informatika dan Profesionalisme 
“Accountability and Workplace Relationships” 
 
 
 
Nama : Tika Altiora Sitanggang 
NIM : 42324025 
Prodi : D-III Teknologi Informasi 
 
 
 
...'


Penjelasan: Kode ini memecah 9 halaman tadi menjadi potongan teks yang lebih kecil memakai `RecursiveCharacterTextSplitter`.

Parameter yang dipakai:
1. `chunk_size=500`: setiap chunk maksimal 500 karakter.
2. `chunk_overlap=50`: 50 karakter di akhir satu chunk diulang di awal chunk berikutnya, supaya kalimat yang terpotong di perbatasan tidak kehilangan konteks.
3. `length_function=len`: panjang teks dihitung berdasarkan jumlah karakter.
4. `separators=["\n\n", "\n", ". ", " ", ""]`: urutan tempat memotong. Splitter mencoba memotong di pergantian paragraf dulu, lalu pergantian baris, lalu akhir kalimat, lalu spasi, dan terakhir per karakter. Jadi potongan sebisa mungkin berhenti di tempat yang wajar, tidak di tengah kata.

`text_splitter.split_documents(documents)` menjalankan proses pemecahan dan menghasilkan daftar chunk. Metadata halaman asli ikut terbawa ke setiap chunk.

Hasilnya, 9 halaman menjadi 36 chunk dengan rata-rata panjang 422 karakter. Rata-ratanya di bawah 500 karena splitter memilih memotong di batas paragraf atau kalimat, bukan memaksa sampai tepat 500 karakter. Pada chunk pertama, `page: 0` artinya halaman pertama, karena penomoran dimulai dari 0.

# 5. Embedding & Vector Database

Tahap ini mengubah setiap chunk teks menjadi **vektor embedding** yang merepresentasikan makna teks tersebut, lalu menyimpannya ke **FAISS** sebagai vector database yang bisa dicari secara semantik.

Contoh: kata yang maknanya mirip --> vektornya berdekatan di ruang vektor
```
"kucing"  → [0.2, 0.8, 0.1, ...]
"anjing"  → [0.3, 0.7, 0.2, ...]  <-- dekat dengan kucing
"laptop"  → [0.9, 0.1, 0.7, ...]  <-- jauh dari kucing
```

**Dua komponen utama:**
1. **Embedding model** (`all-MiniLM-L6-v2`) https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
  > Digunakan untuk mengubah teks menjadi vektor. Model kecil (~80MB), jalan di CPU, dioptimalkan untuk semantic search. Berbeda dengan model LLM, embedding model **tidak generate teks**, hanya menghasilkan representasi numerik dari makna sebuah teks.

2. **FAISS** (Facebook AI Similarity Search) https://github.com/facebookresearch/faiss
  > Digunakan untuk menyimpan & mencari vektor. Mencari chunk yang paling relevan berdasarkan **kemiripan makna**, bukan kesamaan kata. Contoh: query *"cara daftar beasiswa"* bisa menemukan chunk yang berisi *"prosedur pendaftaran"* meskipun kata-katanya berbeda.

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Download & Load embedding model
# Model ini berjalan lokal di Colab (CPU), bukan via API
embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# Model lain bisa dicari di https://huggingface.co/sentence-transformers
# Contoh:
# sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
# sentence-transformers/all-MiniLM-L6-v2
# BAAI/bge-m3

# Buat embedding untuk semua chunks
# Setiap chunk diubah menjadi vektor 384 dimensi
# normalize_embeddings=True --> memungkinkan cosine similarity via inner product
print(f"\nMembuat embedding untuk {len(chunks)} chunks...")
chunk_texts = [c.page_content for c in chunks]
chunk_embeddings = embed_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Buat FAISS index
dim = chunk_embeddings.shape[1]          # dimensi vektor (all-MiniLM-L6-v2)
index = faiss.IndexFlatIP(dim)           # Inner Product = cosine similarity (karena normalized)
index.add(np.array(chunk_embeddings))    # masukkan semua embedding ke index

print(f"\nVector database berhasil dibuat!")
print(f"   Dimensi vektor  : {dim}")
print(f"   Total chunks    : {index.ntotal}")

# Helper: fungsi retrieval
# Mengubah query teks --> vektor, lalu mencari k chunks
# yang paling mirip secara semantik di FAISS index
def retrieve(query: str, k: int = 3):
    """Cari k chunks paling relevan untuk query."""
    query_vec = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_vec), k)
    return [chunks[i] for i in indices[0] if i < len(chunks)]

# Test retrieval
print(f"\nTest retrieval — cari 2 chunk paling relevan:")
test_results = retrieve("What is this document about?", k=2)
for i, doc in enumerate(test_results):
    print(f"\n   Result {i+1} (halaman {doc.metadata.get('page', '?')}):")
    print(f"   '{doc.page_content[:150]}...'")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Membuat embedding untuk 36 chunks...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Vector database berhasil dibuat!
   Dimensi vektor  : 384
   Total chunks    : 36

Test retrieval — cari 2 chunk paling relevan:

   Result 1 (halaman 7):
   '4.6 Text Activity 6: Ukuran Akuntabilitas (Activity 6-4) 
Perintah: Tentukan tugas yang diperlukan untuk mencapai tujuan utama dalam rencana hidup 
An...'

   Result 2 (halaman 0):
   'Laporan Praktikum Socio - Informatika dan Profesionalisme 
“Accountability and Workplace Relationships” 
 
 
 
Nama : Tika Altiora Sitanggang 
NIM : 4...'


Penjelasan: Kode ini mengubah semua chunk menjadi vektor, menyimpannya di FAISS, lalu membuat fungsi pencarian.

1. `SentenceTransformer(...)` memuat embedding model yang berjalan langsung di Colab, bukan lewat API. Karena dokumennya berbahasa Indonesia, model yang dipakai adalah versi multilingual supaya teks bahasa Indonesia dan Inggris sama-sama bisa dipahami.
2. `embed_model.encode(chunk_texts, ...)` mengubah setiap chunk menjadi vektor berukuran 384 angka. Parameter `normalize_embeddings=True` membuat panjang tiap vektor jadi 1, sehingga pengukuran kemiripan bisa memakai inner product yang hasilnya setara dengan cosine similarity.
3. `dim = chunk_embeddings.shape[1]` mengambil jumlah dimensi vektor (384) untuk membuat index.
4. `faiss.IndexFlatIP(dim)` membuat index dengan metode inner product. Metode ini membandingkan query ke semua vektor satu per satu, jadi hasilnya akurat dan cukup cepat untuk 36 chunk.
5. `index.add(...)` memasukkan semua embedding ke index.

Fungsi `retrieve(query, k=3)` adalah inti dari tahap retrieval:
1. Query diubah jadi vektor dengan embedding model yang sama.
2. `index.search(query_vec, k)` mencari k vektor paling mirip dan mengembalikan skor serta nomor index chunk-nya.
3. Nomor index dipakai untuk mengambil chunk aslinya dari list `chunks`. Kondisi `if i < len(chunks)` mencegah error kalau FAISS mengembalikan index kosong.

Pada tes retrieval, pertanyaan "What is this document about?" menemukan chunk dari halaman 7 dan halaman 0 (halaman judul). Walaupun pertanyaannya berbahasa Inggris dan dokumennya berbahasa Indonesia, pencarian tetap jalan karena yang dibandingkan adalah makna, bukan kata yang sama persis.

# 6. Konfigurasi Model LLM

Bagian ini menentukan model LLM yang akan digunakan untuk **generate jawaban**.

> 💡 Model akan didownload dari HuggingFace Hub.
> Download hanya dilakukan **sekali per session**. Jika runtime tidak di-restart, model tetap tersedia.

**Syarat model yang bisa dipakai untuk RAG:**
1. Tersedia di HuggingFace Hub https://huggingface.co/models. Model di luar HuggingFace bisa dipakai tapi perlu di-load manual dari path lokal.
2. Versi **Instruct** atau **Chat**, bukan base model. Base model hanya melanjutkan teks, bukan menjawab pertanyaan. Sedangkan versi Instruct/Chat sudah dilatih khusus untuk mengikuti instruksi sehingga cocok untuk pipeline RAG.
3. Ukuran sesuai VRAM (memori GPU) yang tersedia. Jika ukuran model melebihi VRAM, loading model akan gagal.


In [ ]:
# Ganti MODEL_ID dengan model LLM yang diinginkan
# Contoh: https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

Penjelasan: Di sini kita hanya menentukan model LLM yang akan dipakai lewat variabel `MODEL_ID`. Isinya adalah nama model di HuggingFace Hub dengan format `pemilik/nama-model`.

Model yang dipilih adalah `Qwen/Qwen2.5-1.5B-Instruct`. Alasannya:
1. Ini versi Instruct, jadi sudah dilatih untuk menjawab pertanyaan dan mengikuti perintah, cocok untuk RAG.
2. Ukurannya 1,5 miliar parameter, cukup kecil untuk dimuat di GPU T4 Colab tanpa kehabisan VRAM.
3. Model ini mendukung banyak bahasa, termasuk bahasa Indonesia.

Kalau mau mencoba model lain, cukup ganti `MODEL_ID`, selama ukurannya masih muat di VRAM.

# 7. Load Model & Bangun RAG Chain

Tahap ini melakukan dua hal sekaligus:
1. **Download & load LLM** dari HuggingFace Hub ke memori GPU
2. **Bangun RAG chain** yang menghubungkan semua komponen

In [ ]:
import torch
import transformers
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

# Download & Load Model LLM
# Model di-download dari HuggingFace Hub ke Colab session
# device_map="auto" --> otomatis load ke GPU jika tersedia, fallback ke CPU
pipe = transformers.pipeline(
    "text-generation",
    model=MODEL_ID,
    model_kwargs={
        "torch_dtype": torch.bfloat16 if DEVICE == "cuda" else torch.float32
    },
    device_map="auto",
)

print(f"Model berhasil di-load!")
print(f"Model ID : {MODEL_ID}")

# Fungsi generate jawaban
# Menerima konteks + pertanyaan, mengembalikan jawaban dari LLM
def call_llm(prompt_text: str) -> str:
    """Generate jawaban dari model lokal."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer based ONLY on the provided context. Be concise."},
        {"role": "user",   "content": prompt_text},
    ]
    # max_new_tokens --> Jumlah maksimum token yang boleh di-generate model
    # temperature --> Mengontrol "kreativitas" / keacakan output
    # do_sample --> True agar temperature berlaku; False = greedy decoding (selalu pilih token probabilitas tertinggi)
    outputs = pipe(
        messages,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
    )
    # Ambil hanya teks jawaban terakhir (bukan history)
    return outputs[0]["generated_text"][-1]["content"].strip()

# Prompt Augmentation
# Instruksi (prompt) eksplisit agar model LLM hanya menjawab dari konteks
# {context} dan {question} diisi otomatis oleh RAG chain
RAG_PROMPT = """Answer the question based ONLY on the context below.
If not found in context, say \"I don't have enough information.\"

Context:
{context}

Question: {question}"""

prompt = PromptTemplate(
    template=RAG_PROMPT,
    input_variables=["context", "question"]
)

# Helper: retrieve + format konteks
# Menggabungkan hasil retrieve() menjadi satu blok teks
# dengan label halaman sebagai referensi sumber
def format_docs(question: str) -> str:
    docs = retrieve(question, k=3)
    return "\n\n".join([
        f"[Halaman {doc.metadata.get('page', '?')}]\n{doc.page_content}"
        for doc in docs
    ])

# Bangun RAG Chain Pipeline
# Menghubungkan semua komponen menjadi satu pipeline
def rag_pipeline(question: str) -> str:
    context = format_docs(question)
    filled_prompt = prompt.format(context=context, question=question)
    return call_llm(filled_prompt)

rag_chain = RunnableLambda(rag_pipeline)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model berhasil di-load!
Model ID : Qwen/Qwen2.5-1.5B-Instruct


Penjelasan: Sel ini punya dua tugas, yaitu memuat LLM lalu menyusun alur RAG dari semua komponen sebelumnya.

Load model:
1. `transformers.pipeline("text-generation", ...)` membuat pipeline untuk menghasilkan teks dari model `MODEL_ID`.
2. `torch_dtype` menentukan presisi angka model. `bfloat16` dipakai di GPU supaya hemat memori, dan `float32` dipakai di CPU. Pilihannya mengikuti variabel `DEVICE`.
3. `device_map="auto"` membuat model otomatis ditaruh di GPU kalau tersedia.

Fungsi `call_llm(prompt_text)`:
1. Pesan disusun dalam format chat. Role `system` berisi aturan umum untuk model, dan role `user` berisi prompt yang sudah lengkap dengan konteks dan pertanyaan.
2. `max_new_tokens=512` membatasi panjang jawaban maksimal 512 token.
3. `temperature=0.3` membuat jawaban lebih stabil dan tidak terlalu acak. Nilai kecil cocok untuk RAG karena jawaban harus sesuai dokumen.
4. `do_sample=True` diperlukan agar pengaturan temperature berlaku. Kalau `False`, model memilih token paling mungkin saja (greedy decoding).
5. `outputs[0]["generated_text"][-1]["content"]` mengambil teks jawaban terakhir saja, tanpa riwayat pesan sebelumnya.

Bagian prompt (augmentation):
1. `RAG_PROMPT` adalah template instruksi yang menyuruh model menjawab hanya berdasarkan konteks. Kalau jawabannya tidak ada, model disuruh bilang tidak punya cukup informasi. Ini mengurangi risiko model mengarang jawaban (halusinasi).
2. `{context}` dan `{question}` adalah placeholder yang nanti diisi otomatis.
3. `PromptTemplate` membungkus template tersebut supaya mudah diisi lewat `prompt.format(...)`.

Fungsi pendukung dan chain:
1. `format_docs(question)` memanggil `retrieve` dengan k=3, lalu menggabungkan tiga chunk jadi satu teks. Setiap chunk diberi label halaman sebagai referensi sumber.
2. `rag_pipeline(question)` menjalankan urutan lengkapnya: ambil konteks, isi prompt, lalu kirim ke LLM.
3. `RunnableLambda(rag_pipeline)` membungkus fungsi tadi jadi objek chain LangChain, sehingga bisa dipanggil dengan `rag_chain.invoke(pertanyaan)`.

Output "Model berhasil di-load!" menandakan model siap dipakai.

# 8. Test RAG Chain

Sebelum membungkus dengan UI, uji RAG chain langsung di console.
Output verbose menampilkan chunks yang ditemukan retriever — berguna untuk
memastikan retrieval bekerja dengan benar sebelum lanjut ke Gradio.

**Ganti pertanyaan** di `test_questions` sesuai isi PDF yang dipakai.

In [ ]:
# Fungsi untuk bertanya ke RAG chain.
# verbose=True  --> tampilkan chunks yang ditemukan + jawaban
# verbose=False --> hanya kembalikan jawaban (dipakai untuk inference di Gradio)
def ask(question: str, verbose: bool = True):
    """
    Fungsi utama untuk bertanya ke chatbot.
    Set verbose=True untuk lihat chunks yang diambil.
    """
    if verbose:
        print(f"Pertanyaan: {question}")

        # Tampilkan chunks yang ditemukan retriever
        # Berguna untuk debug. Apakah retrieval sudah relevan?
        print("\nChunks yang ditemukan (retrieval):")
        retrieved = retrieve(question, k=3)   # pakai fungsi retrieve()
        for i, doc in enumerate(retrieved):
            print(f"  [{i+1}] Halaman {doc.metadata.get('page','?')}: '{doc.page_content[:100]}...'")
        print()

    answer = rag_chain.invoke(question)
    answer = answer.strip()

    if verbose:
        print(f"Jawaban:")
        print(f"   {answer}")
        print("-" * 60)

    return answer

Penjelasan: Fungsi `ask(question, verbose=True)` dibuat untuk menguji RAG chain dengan mudah.

1. Kalau `verbose=True`, fungsi menampilkan pertanyaan dan tiga chunk hasil retrieval (lengkap dengan nomor halaman) sebelum menampilkan jawaban. Ini berguna untuk debugging, karena kita bisa melihat apakah chunk yang diambil memang relevan dengan pertanyaan.
2. `rag_chain.invoke(question)` menjalankan seluruh pipeline RAG dan mengembalikan jawaban dari LLM. `.strip()` membuang spasi atau baris kosong di awal dan akhir.
3. Kalau `verbose=False`, fungsi hanya mengembalikan jawaban tanpa mencetak apa pun. Mode ini dipakai nanti di Gradio.

In [ ]:
# Ganti pertanyaan berikut sesuai isi PDF
test_questions = [
    "What is the main topic of this document?",
    "Can you summarize the key points?",
    # Tambahkan pertanyaan spesifik sesuai isi PDF
    "What are the negative effects of talking badly about others during networking?",
    "How should Julian handle the gossip situation involving Evan?",
    "What specific accountability measures are set for achieving the life goal of graduating on time with a cumlaude predicate?",
    "What additional disruptions should employees avoid in shared workspaces?",
    "What steps can Christine take to show that she is responsible and accountable?",
    "How should Monique handle Josh's cold attitude towards her?",
    "How can a manager motivate employees to take on additional responsibilities?",
    "What should a manager do if the morale of employees in their department decreases?",
    "What actions should you take if a family emergency occurs the night before an important meeting?",
    "What potential problems can arise if you date a new sales representative who services your office equipment?"
]

for q in test_questions:
    ask(q, verbose=True)

Pertanyaan: What is the main topic of this document?

Chunks yang ditemukan (retrieval):
  [1] Halaman 4: 'baik, dan kesempatan belajar sering lebih berpengaruh. 
4.2 Text Activity 2: Masalah di Tempat Kerja...'
  [2] Halaman 7: '4.6 Text Activity 6: Ukuran Akuntabilitas (Activity 6-4) 
Perintah: Tentukan tugas yang diperlukan u...'
  [3] Halaman 3: 'kedua, saya bertanya dengan pertanyaan terbuka, seperti “Apa yang membuat kamu nyaman di 
pekerjaan ...'

Jawaban:
   The main topic of this document appears to be workplace issues and conflict management strategies.
------------------------------------------------------------
Pertanyaan: Can you summarize the key points?

Chunks yang ditemukan (retrieval):
  [1] Halaman 7: '4.6 Text Activity 6: Ukuran Akuntabilitas (Activity 6-4) 
Perintah: Tentukan tugas yang diperlukan u...'
  [2] Halaman 3: 'kedua, saya bertanya dengan pertanyaan terbuka, seperti “Apa yang membuat kamu nyaman di 
pekerjaan ...'
  [3] Halaman 5: '4. Saya menjelaskan sit

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Jawaban:
   A manager should observe specific behaviors indicating decreased employee morale, such as tardiness or declining work quality. Once identified, they can address it by addressing the root cause and encouraging positive behavior through clear communication and professional conduct expectations. If necessary, they may also involve HR to mediate between conflicting colleagues.
------------------------------------------------------------
Pertanyaan: What actions should you take if a family emergency occurs the night before an important meeting?

Chunks yang ditemukan (retrieval):
  [1] Halaman 5: 'Jawaban: 
Prioritas pertama saya adalah menolong anggota keluarga inti saya. Jika keadaan daruratnya...'
  [2] Halaman 5: 'tidak adil. 
Tidak ada insentif dan 
penghargaan 
Kerja baik dan kerja asal-asalan diperlakukan sama...'
  [3] Halaman 5: '4. Saya menjelaskan situasinya dan memberikan informasi atau bahan rapat yang dibutuhkan, 
supaya or...'

Jawaban:
   Saya menjelaskan situasi

Penjelasan: Sel ini menjalankan `ask()` untuk semua pertanyaan di list `test_questions` lewat perulangan `for`. Dua pertanyaan pertama bersifat umum, sedangkan sisanya sengaja dibuat spesifik sesuai isi PDF, misalnya tentang kasus Julian dan Evan, Christine, atau Monique dan Josh.

Dari output terlihat bahwa kualitas jawaban sangat bergantung pada chunk yang berhasil diambil retriever. Pada pertanyaan terakhir, ketiga chunk berasal dari halaman 6 dan jawabannya nyambung dengan isi chunk tersebut. Karena LLM yang dipakai berukuran kecil, sebagian jawaban bisa singkat atau kurang lengkap. Kalau jawabannya kurang tepat, hal pertama yang dicek adalah chunk hasil retrieval, apakah chunk yang relevan memang ikut terambil.

# 9. Gradio Chatbot UI
Tahap terakhir: membungkus pipeline RAG chain ke dalam antarmuka chatbot yang bisa diakses siapapun melalui browser.

**Cara kerja Gradio di Google Colab:**
- Gradio membuat **local server** di dalam Colab session
- Parameter `share=True` menghasilkan **public link** (tunnel ke server Gradio). Link aktif selama **72 jam** atau sampai session Google Colab ditutup

> Public link bisa dibagikan ke siapapun — tidak perlu login atau install apapun. Cocok untuk demo MVP yang langsung bisa dicoba.

In [ ]:
import gradio as gr
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

DOCUMENT_NAME = "Accountability_42324025"  # Ganti sesuai nama PDF

# Fungsi handler Gradio
# Dipanggil setiap kali user mengirim pesan
# history: list pesan sebelumnya (dikelola Gradio otomatis)
def chat_with_doc(message: str, history: list):
    if not message.strip():
        return "Silakan ketik pertanyaan Anda"
    try:
        return rag_chain.invoke(message)
    except Exception as e:
        return f"Error: {str(e)}"

# Ganti contoh pertanyaan sesuai isi PDF
example_questions = [
    ["What is this document about?"],
    ["What are the main topics covered?"],
    ["Give me a brief summary."],
]

# Bangun UI
with gr.Blocks(title=f"RAG Chatbot — {DOCUMENT_NAME}") as demo:

    gr.Markdown(f"""
    # RAG Chatbot — {DOCUMENT_NAME}
    Tanya apapun tentang dokumen ini!
    > **Tips:** Selalu gunakan pertanyaan spesifik untuk hasil terbaik.
    """)

    gr.ChatInterface(
        fn=chat_with_doc,
        chatbot=gr.Chatbot(
            height=450,
            allow_tags=False,
            placeholder="<strong>Halo! Saya siap menjawab pertanyaan Anda.</strong>",
            show_label=False,
        ),
        textbox=gr.Textbox(
            placeholder="Ketik pertanyaan Anda di sini...",
            container=False,
            scale=7
        ),
        examples=example_questions,
    )

    gr.Markdown(f"""
    ---
    **Model:** {MODEL_ID.split('/')[-1]} | **Embedding:** all-MiniLM-L6-v2 | **Vector DB:** FAISS
    """)

# Jalankan Gradio
# share=True  --> generate public link yang bisa dibagikan
# debug=False --> sembunyikan error internal dari user
# quiet=True  --> suppress log Gradio yang tidak perlu
demo.launch(share=True, debug=False, quiet=True)

* Running on public URL: https://525fb9631e359bcf61.gradio.live


Penjelasan: Kode ini membungkus RAG chain menjadi chatbot berbasis web memakai Gradio.

1. `warnings.filterwarnings(...)` menyembunyikan peringatan deprecation supaya output lebih bersih.
2. `DOCUMENT_NAME` adalah nama dokumen yang ditampilkan di judul chatbot.
3. `chat_with_doc(message, history)` adalah fungsi yang dipanggil setiap kali pengguna mengirim pesan. Kalau pesan kosong, fungsi mengembalikan pengingat untuk mengetik pertanyaan. Kalau tidak, pesan dikirim ke `rag_chain.invoke(message)`. Blok `try/except` menangkap error supaya aplikasi tidak langsung berhenti dan pengguna tetap melihat pesan kesalahan.
4. `example_questions` berisi contoh pertanyaan yang muncul sebagai tombol cepat di UI.
5. `gr.Blocks` adalah wadah untuk menyusun tampilan. `gr.Markdown` menampilkan judul, tips, dan info model di bagian bawah.
6. `gr.ChatInterface` membuat tampilan chat lengkap. Parameter `fn` diisi fungsi `chat_with_doc`, `chatbot` mengatur kotak percakapan (tinggi 450 piksel dan pesan sambutan), dan `textbox` mengatur kolom input.
7. `demo.launch(share=True)` menjalankan aplikasi dan membuat public link yang bisa dibuka siapa saja selama sesi Colab masih aktif. Parameter `debug=False` menyembunyikan error internal dari pengguna, dan `quiet=True` mengurangi log yang tidak perlu.

Pada output, chatbot sudah berjalan di public URL dan bisa menjawab pertanyaan "What is this document about?" dengan ringkasan isi dokumen.

In [ ]:
demo.close()

Closing server running on port: 7860


Penjelasan: `demo.close()` menghentikan server Gradio yang berjalan di Colab, dalam output ini di port 7860. Public link juga ikut mati setelah server ditutup. Sel ini dijalankan kalau demo sudah selesai atau kalau mau meluncurkan ulang Gradio tanpa error port yang masih dipakai.

# 10. Opsional: Ganti Dokumen PDF tanpa Restart Session

Tahap ini melakukan re-indexing dokumen baru secara langsung. Model LLM tidak perlu di-load ulang karena model tetap sama, hanya vector database-nya yang diganti:
1. Upload PDF baru
2. Load & chunk dokumen baru
3. Buat embedding & FAISS index baru (menggantikan yang lama)
4. RAG chain otomatis terhubung ke dokumen baru


In [ ]:
import numpy as np

# Upload PDF baru
print("Upload PDF baru...")
uploaded_new = files.upload()
new_pdf = list(uploaded_new.keys())[0]

# Load & ekstrak PDF baru
reader_new = PdfReader(new_pdf)
new_docs = []
for page_num, page in enumerate(reader_new.pages):
    text = page.extract_text()
    if text and text.strip():
        new_docs.append(Document(
            page_content=text,
            metadata={"page": page_num, "source": new_pdf}
        ))

# Chunk ulang
new_chunks = text_splitter.split_documents(new_docs)
print(f"{len(reader_new.pages)} halaman --> {len(new_chunks)} chunks")

# Re-embedding & rebuild FAISS index
# LLM tidak perlu di-load ulang. Hanya vector database yang diganti
new_texts = [c.page_content for c in new_chunks]
new_embeddings = embed_model.encode(
    new_texts,
    normalize_embeddings=True,
    show_progress_bar=True)

# Update variabel global index & chunks
# agar retrieve() otomatis pakai dokumen baru
dim = new_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.array(new_embeddings))
chunks = new_chunks

print(f"Berhasil ganti ke dokumen: {new_pdf}")
print(f"Total chunks baru: {index.ntotal}")

Upload PDF baru...


Saving Ethics_42324025.pdf to Ethics_42324025.pdf
12 halaman --> 54 chunks


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Berhasil ganti ke dokumen: Ethics_42324025.pdf
Total chunks baru: 54


Penjelasan: Sel ini dipakai untuk mengganti dokumen tanpa memuat ulang model LLM yang beratnya besar. Langkahnya sama dengan tahap awal, hanya memakai variabel baru supaya data lama tidak tertimpa sebelum proses selesai.

1. `files.upload()` meminta upload PDF baru, lalu nama filenya diambil seperti sebelumnya.
2. PDF dibaca halaman per halaman dan dijadikan `new_docs`, dengan metadata halaman dan sumber yang baru.
3. `text_splitter.split_documents(new_docs)` memecah dokumen baru memakai splitter yang sama, dengan pengaturan chunk yang sama.
4. `embed_model.encode(new_texts, ...)` membuat embedding untuk chunk baru memakai embedding model yang sudah dimuat, jadi tidak perlu download ulang.

Setelah embedding dibuat, index FAISS dan list `chunks` perlu diganti dengan yang baru. Karena fungsi `retrieve` membaca variabel `index` dan `chunks`, begitu keduanya diganti, RAG chain otomatis menjawab berdasarkan dokumen baru. LLM tidak disentuh sama sekali, jadi prosesnya cepat.